In [1]:
import pandas as pd

In [10]:
import requests
import re
import json
import pandas as pd
from bs4 import BeautifulSoup

def get_holdings_comprehensive(ticker):
    """
    Comprehensive holdings fetcher with multiple fallback methods.
    Priority: yfinance > ETFdb > iShares > Vanguard
    """
    
    # METHOD 1: yfinance (fastest and most reliable)
    try:
        import yfinance as yf
        etf = yf.Ticker(ticker)
        holdings = etf.get_holdings()
        if holdings is not None and not holdings.empty:
            print(f"✅ yfinance: Found {len(holdings)} holdings")
            return holdings.head(10)
    except:
        pass
    
    # METHOD 2: ETFdb.com (good alternative)
    try:
        url = f"https://etfdb.com/etf/{ticker}/#holdings"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }
        r = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        
        # Find holdings table
        table = soup.find('table', {'class': 'table'})
        if table:
            df = pd.read_html(str(table))[0]
            print(f"✅ ETFdb.com: Found {len(df)} holdings")
            return df.head(10)
    except Exception as e:
        print(f"⚠️ ETFdb failed: {e}")
    
    # METHOD 3: iShares specific (for BlackRock ETFs like IVV, AGG)
    if ticker.upper() in ['IVV', 'AGG', 'IJH', 'IJR', 'EFA', 'EMB', 'LQD', 'TIP']:
        try:
            search_url = f"https://www.ishares.com/us/products/etf-investments"
            # This requires more complex scraping - simplified version
            print(f"⚠️ iShares ETF detected - manual lookup recommended")
        except:
            pass
    
    # METHOD 4: Return None if all methods fail
    print(f"❌ Could not retrieve holdings for {ticker}")
    return None


# Test function
test_ticker = "VOO"
print(f"Fetching holdings for {test_ticker}...\n")
holdings_df = get_holdings_comprehensive(test_ticker)

if holdings_df is not None:
    print("\nTop Holdings:")
    print(holdings_df)

Fetching holdings for VOO...

✅ ETFdb.com: Found 4 holdings

Top Holdings:
                   Type Symbol Expense Ratio    Assets Avg. Daily Vol  \
0              Cheapest   BKLC         0.00%    $4.9 B         149066   
1         Largest (AUM)    IVV         0.03%  $729.9 B            8 M   
2  Most Liquid (Volume)    SPY         0.09%  $701.5 B           79 M   
3     Top YTD Performer   ARKW         0.76%    $2.2 B         239809   

  YTD Return  
0     18.22%  
1     17.93%  
2     17.80%  
3     44.63%  
✅ ETFdb.com: Found 4 holdings

Top Holdings:
                   Type Symbol Expense Ratio    Assets Avg. Daily Vol  \
0              Cheapest   BKLC         0.00%    $4.9 B         149066   
1         Largest (AUM)    IVV         0.03%  $729.9 B            8 M   
2  Most Liquid (Volume)    SPY         0.09%  $701.5 B           79 M   
3     Top YTD Performer   ARKW         0.76%    $2.2 B         239809   

  YTD Return  
0     18.22%  
1     17.93%  
2     17.80%  
3     44.63% 

/var/folders/km/_b5m00hj0l91hg5vzm9rf92m0000gn/T/ipykernel_15538/1509551450.py:36: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]
